# Wikipedia Tech Company Revenue Scraper

This notebook scrapes three historical revenue tables (2021, 2023, 2025) for the largest technology companies from Wikipedia, and saves each table as a separate CSV file for later loading into the SQL Server warehouse (Bronze layer).

In [ ]:
# Import libraries
# - requests: fetch the raw HTML of the Wikipedia page
# - BeautifulSoup: parse the HTML and find/extract table data
# - pandas: build clean DataFrames from the scraped data and export to CSV
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [ ]:
# Show up to 500 rows when previewing DataFrames (avoids pandas truncating output)
pd.set_option('display.max_rows', 500)

## Step 1: Request the Wikipedia page

In [ ]:
# Target page: List of largest technology companies by revenue
url = 'https://en.wikipedia.org/wiki/List_of_largest_technology_companies_by_revenue'

# Wikipedia requires a descriptive User-Agent header identifying the tool/contact
# email, otherwise requests may be throttled or blocked
headers = {
    "User-Agent": "WikiDataScraper/1.0 (suleman.safdar975@gmail.com)"
}

# Download the raw HTML of the page
page = requests.get(url, headers=headers)

## Step 2: Parse the HTML

In [ ]:
# Parse the page with BeautifulSoup
response = BeautifulSoup(page.text, 'html')

# The page contains multiple tables; index [1] is the current-year (2025) revenue table.
# NOTE: table indices depend on Wikipedia's current page layout, so if the page
# structure changes, these indices may need to be updated.
table = response.find_all('table')[1]

## Step 3: Extract column headers (2025 table)

In [ ]:
# Grab all <th> header cells from the table
heading = table.find_all('th')

In [ ]:
# Clean header text (strip whitespace/newlines) to use as DataFrame column names
titles = [title.text.strip() for title in heading]
titles

## Step 4: Create an empty DataFrame with the scraped column names

In [ ]:
df = pd.DataFrame(columns=titles)
df

## Step 5: Extract row data (2025 table) and populate the DataFrame

In [ ]:
# Loop through every table row, skipping the header row ([1:])
column_data = table.find_all('tr')
for row in column_data[1:]:
    row_data = row.find_all('td')          # grab all data cells in the row
    data = [data.text.strip() for data in row_data]   # clean text
    df.loc[len(df)] = data                  # append as a new row

In [ ]:
# Save the 2025 revenue table to CSV (Bronze source file #1)
df.to_csv('tech_acompany_info.csv', index=False)

## Step 6: Repeat the process for the previous two years' tables (2023 and 2021)

In [ ]:
# Table index [2] on the page corresponds to the 2023 revenue table
table1 = response.find_all('table')[2]

In [ ]:
# Extract headers and build an empty DataFrame for the 2023 table
heading1 = table1.find_all('th')
title1 = [title.text.strip() for title in heading1]
df1 = pd.DataFrame(columns=title1)

In [ ]:
# Populate the 2023 DataFrame the same way as the 2025 table
data_column = table1.find_all('tr')
for row in data_column[1:]:
    row_data1 = row.find_all('td')
    data1 = [data.text.strip() for data in row_data1]
    df1.loc[len(df1)] = data1

In [ ]:
# Table index [3] on the page corresponds to the 2021 revenue table
table2 = response.find_all('table')[3]
heading2 = table2.find_all('th')
title2 = [title2.text.strip() for title2 in heading2]

# The 2021 table has an extra leading header (e.g. rank/index) that has no
# matching data column, so it's dropped here to keep headers and data aligned
title2 = title2[1:]

In [ ]:
df2 = pd.DataFrame(columns=title2)
df2

In [ ]:
# Populate the 2021 DataFrame
# [2:] skips the first two <td> cells in each row (rank + a duplicate/empty cell)
# to line up with the trimmed title2 headers above
data_column2 = table2.find_all('tr')
for row in data_column2[1:]:
    row_data2 = row.find_all('td')[2:]
    data2 = [data2.text.strip() for data2 in row_data2]
    df2.loc[len(df2)] = data2

## Step 7: Export the 2023 and 2021 tables to CSV (Bronze source files #2 and #3)

In [ ]:
df1.to_csv('tech_company_info2.csv', index=False)   # 2023 data
df2.to_csv('tech_company_info3.csv', index=False)   # 2021 data